# 05. Common Feature Pipeline

## 목적

새 연구 설계에서 사용할 공통 Point-in-Time 데이터셋을 구축한다.

전체 흐름:

KRX300 Historical Membership
→ Stock / Index Raw Data
→ Cleaning
→ Point-in-Time Alignment
→ Common Features
→ Model Input Dataset

### Data Layers

- `data/raw` : 원본 데이터
- `data/clean` : 정제 데이터
- `data/aligned` : 날짜 및 Universe 정렬 데이터
- `data/features` : 최종 파생 Feature

### 핵심 원칙

1. 현재 구성종목을 과거에 소급하지 않는다.
2. 날짜 t 종가 기반 정보는 t+1 이전 의사결정에 사용하지 않는다.
3. Raw 데이터는 수정하지 않는다.
4. 모든 모델이 동일한 Point-in-Time Information Set을 공유한다.

In [1]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd

from dotenv import load_dotenv

In [2]:
PROJECT_ROOT = Path(r"C:\code\portfolio_optimization")

DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
CLEAN_DIR = DATA_DIR / "clean"
ALIGNED_DIR = DATA_DIR / "aligned"
FEATURE_DIR = DATA_DIR / "features"

KRX_UNIVERSE_DIR = RAW_DIR / "krx" / "universe"
KRX_STOCK_DIR = RAW_DIR / "krx" / "stocks"
KRX_INDEX_DIR = RAW_DIR / "krx" / "index"

load_dotenv(
    PROJECT_ROOT / ".env"
)

print(PROJECT_ROOT)

C:\code\portfolio_optimization


In [3]:
print(
    "KRX ID loaded:",
    bool(os.getenv("KRX_ID"))
)

print(
    "KRX PW loaded:",
    bool(os.getenv("KRX_PW"))
)

print(
    "KRX API key loaded:",
    bool(os.getenv("KRX_API_KEY"))
)

KRX ID loaded: True
KRX PW loaded: True
KRX API key loaded: True


from pykrx import stock

KRX300_CODE = "5300"

print(
    stock.get_index_ticker_name(
        KRX300_CODE
    )
)

#point-in-time membership 함수

def get_krx300_members(date):
    """
    특정 날짜의 실제 KRX300 구성종목을 조회

    Parameters
    ----------
    date : str
        YYYYMMDD 형식

    Returns
    -------
    list
        해당 날짜의 KRX300 종목코드 목록
    """

    members = (
        stock.get_index_portfolio_deposit_file(
            KRX300_CODE,
            date
        )
    )

    return list(members)

In [7]:
#test

members_2026 = get_krx300_members(
    "20260915"
)

print(
    len(members_2026)
)

print(
    members_2026[:10]
)

301
['005930', '000660', '402340', '009150', '373220', '005380', '207940', '105560', '032830', '028260']


In [8]:
#월별 검사 날짜

#날짜 범위
UNIVERSE_START = "2018-02-05"
UNIVERSE_END = "2026-09-15"

#월말 기준
monthly_dates = pd.date_range(
    start=UNIVERSE_START,
    end=UNIVERSE_END,
    freq="ME"
)

monthly_dates[:5]

DatetimeIndex(['2018-02-28', '2018-03-31', '2018-04-30', '2018-05-31',
               '2018-06-30'],
              dtype='datetime64[ns]', freq='ME')

In [9]:
print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "KRX300:",
    stock.get_index_ticker_name(
        KRX300_CODE
    )
)

print(
    "2026-09-15 members:",
    len(
        get_krx300_members(
            "20260915"
        )
    )
)

print(
    "2018-02-05 members:",
    len(
        get_krx300_members(
            "20180205"
        )
    )
)

Project root: C:\code\portfolio_optimization
KRX300: KRX 300
2026-09-15 members: 301
2018-02-05 members: 305


In [10]:
#05a-1
# 원천 데이터 수집 1
# KRX300 지수 데이터를 이용해 실제 거래일 trading calendar 생성

# 목적:
# - 달력 날짜가 아니라 실제 한국거래소 거래일 기준으로 모든 데이터를 정렬
# - 이후 KRX300 구성종목 snapshot 날짜 선정에 사용

KRX300_INDEX_START = "20180205"
KRX300_INDEX_END = "20260915"

krx300_index_raw = stock.get_index_ohlcv_by_date(
    KRX300_INDEX_START,
    KRX300_INDEX_END,
    KRX300_CODE
)

print(
    "Shape:",
    krx300_index_raw.shape
)

print(
    "Start:",
    krx300_index_raw.index.min()
)

print(
    "End:",
    krx300_index_raw.index.max()
)

krx300_index_raw.head()

Shape: (2112, 7)
Start: 2018-02-05 00:00:00
End: 2026-09-15 00:00:00


KRX 300,시가,고가,저가,종가,거래량,거래대금,상장시가총액
날짜,,,,,,,
2018-02-05,1487.99,1498.10,1480.99,1489.41,143693847,8263559557872,1578528023269790
2018-02-06,1455.20,1473.56,1439.81,1468.49,199372022,10371473651627,1556773587022210
2018-02-07,1488.41,1488.61,1429.82,1429.82,154698671,9298597786804,1516597823270820
2018-02-08,1432.08,1451.05,1429.19,1441.13,158769041,10430958812463,1526579124029925
2018-02-09,1402.01,1419.65,1401.73,1411.90,138449265,8212915363740,1496439968175220


In [11]:
#05a-2. trading calendar 분리
#데이터 정제 준비 1
# KRX300 지수 데이터에서 실제 거래일만 추출

# 목적:
# - 휴일 / 주말 제거
# - 이후 모든 데이터의 날짜 기준으로 사용

trading_calendar = pd.DatetimeIndex(
    krx300_index_raw.index
).sort_values()

print(
    "Number of trading days:",
    len(trading_calendar)
)

print(
    "First 5:"
)

print(
    trading_calendar[:5]
)

print(
    "\nLast 5:"
)

print(
    trading_calendar[-5:]
)

Number of trading days: 2112
First 5:
DatetimeIndex(['2018-02-05', '2018-02-06', '2018-02-07', '2018-02-08',
               '2018-02-09'],
              dtype='datetime64[ns]', name='날짜', freq=None)

Last 5:
DatetimeIndex(['2026-09-09', '2026-09-10', '2026-09-11', '2026-09-14',
               '2026-09-15'],
              dtype='datetime64[ns]', name='날짜', freq=None)


In [12]:
# 05a-3. raw data 저장
# KRX300 지수 원본 데이터 저장
# Raw Layer 원칙: 가능하면 가공하지 않고 원본 형태 유지

krx300_index_save = (
    krx300_index_raw
    .reset_index()
)

krx300_index_save.to_csv(
    KRX_INDEX_DIR / "krx300_index_daily_pykrx.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    KRX_INDEX_DIR / "krx300_index_daily_pykrx.csv"
)

Saved: C:\code\portfolio_optimization\data\raw\krx\index\krx300_index_daily_pykrx.csv


In [13]:
#05a-4
# 데이터 정제 준비 2
# 각 월의 마지막 실제 거래일 추출

# 목적:
# - KRX300 구성종목 월별 Snapshot 날짜 선정
# - 주말 / 휴장일 문제 제거

calendar_df = pd.DataFrame({
    "date": trading_calendar
})

calendar_df["month"] = (
    calendar_df["date"]
    .dt
    .to_period("M")
)

month_end_trading_dates = (
    calendar_df
    .groupby("month")["date"]
    .max()
)

print(
    "Number of months:",
    len(month_end_trading_dates)
)

month_end_trading_dates.head(10)

Number of months: 104


month
2018-02   2018-02-28
2018-03   2018-03-30
2018-04   2018-04-30
2018-05   2018-05-31
2018-06   2018-06-29
2018-07   2018-07-31
2018-08   2018-08-31
2018-09   2018-09-28
2018-10   2018-10-31
2018-11   2018-11-30
Freq: M, Name: date, dtype: datetime64[ns]

# 실행 금지 - KRX 자동조회

# 05a-5. 원천 데이터 수집 2
# KRX300 월말 구성종목 Snapshot 수집
# 목적: KRX300 membership 변화가 발생한 구간 탐색
# 주의:이건 최종 Point-in-Time Universe가 아님, 이후 변경 구간의 정확한 변경일을 추가 탐색할 예정

monthly_membership_records = []


for date in month_end_trading_dates:

    date_str = date.strftime(
        "%Y%m%d"
    )

    members = get_krx300_members(
        date_str
    )

    for ticker in members:

        monthly_membership_records.append({
            "date": date,
            "ticker": ticker,
            "index_code": KRX300_CODE,
            "index_name": "KRX 300",
            "member_count": len(members)
        })

monthly_membership = pd.DataFrame(
    monthly_membership_records
)

print(
    "Shape:",
    monthly_membership.shape
)

monthly_membership.head()

In [16]:
#05A-5. KRX300 월말 membership snapshot 로드
# 중요:
# - KRX 웹/API를 새로 호출하지 않음
# - 과거에 수집해 저장한 Snapshot CSV만 사용
# - 자동화 대량 조회 재발 방지

from pathlib import Path
import pandas as pd


PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)

MONTHLY_MEMBERSHIP_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "krx"
    / "universe"
    / "krx300_monthly_membership_snapshots.csv"
)


if not MONTHLY_MEMBERSHIP_PATH.exists():

    raise FileNotFoundError(
        "기존 KRX300 Membership Snapshot 파일이 없습니다. "
        "KRX에 재요청하지 말고 데이터 확보 방식을 다시 설계해야 합니다."
    )


monthly_membership = pd.read_csv(
    MONTHLY_MEMBERSHIP_PATH,
    dtype={
        "ticker": str,
        "index_code": str
    }
)

monthly_membership["date"] = pd.to_datetime(
    monthly_membership["date"]
)


print(
    "Loaded from local cache:",
    MONTHLY_MEMBERSHIP_PATH
)

print(
    "Shape:",
    monthly_membership.shape
)

print(
    "Start:",
    monthly_membership["date"].min()
)

print(
    "End:",
    monthly_membership["date"].max()
)

monthly_membership.head()

Loaded from local cache: C:\code\portfolio_optimization\data\raw\krx\universe\krx300_monthly_membership_snapshots.csv
Shape: (31232, 5)
Start: 2018-02-28 00:00:00
End: 2026-09-15 00:00:00


,date,ticker,index_code,index_name,member_count
0,2018-02-28,005930,5300,KRX 300,305
1,2018-02-28,000660,5300,KRX 300,305
2,2018-02-28,068270,5300,KRX 300,305
3,2018-02-28,005380,5300,KRX 300,305
4,2018-02-28,005490,5300,KRX 300,305


In [ ]:
# 원천 데이터 저장
# 월별 KRX300 구성종목 Snapshot 저장

monthly_membership.to_csv(
    KRX_UNIVERSE_DIR
    / "krx300_monthly_membership_snapshots.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    KRX_UNIVERSE_DIR
    / "krx300_monthly_membership_snapshots.csv"
)

In [ ]:
#05a-6. sanity check
# 데이터 검증 1
# 월별 KRX300 구성종목 수 확인
#목적:구성종목 조회 실패 여부, 비정상적인 0개 / 극단적으로 적은 개수 탐지

monthly_member_counts = (
    monthly_membership
    .groupby("date")["ticker"]
    .nunique()
)

monthly_member_counts.describe()

In [ ]:
monthly_member_counts.head(20)

In [ ]:
monthly_member_counts.tail(20)

In [ ]:
#05a-7. 월간 membership 변경량 계산
# 데이터 검증 2
# 연속된 월 사이 KRX300 구성종목 변경량 계산
# 목적:어느 월 구간에서 종목 편입 / 제외가 발생했는지 탐지

membership_by_date = {
    date: set(
        group["ticker"]
    )
    for date, group
    in monthly_membership.groupby("date")
}

snapshot_dates = sorted(
    membership_by_date.keys()
)

change_records = []

for previous_date, current_date in zip(
    snapshot_dates[:-1],
    snapshot_dates[1:]
):

    previous_members = (
        membership_by_date[
            previous_date
        ]
    )

    current_members = (
        membership_by_date[
            current_date
        ]
    )

    added = (
        current_members
        - previous_members
    )

    removed = (
        previous_members
        - current_members
    )

    change_records.append({
        "previous_date": previous_date,
        "current_date": current_date,
        "added_count": len(added),
        "removed_count": len(removed),
        "changed": (
            len(added) > 0
            or len(removed) > 0
        )
    })

monthly_changes = pd.DataFrame(
    change_records
)

monthly_changes[
    monthly_changes["changed"]
].head(20)

In [ ]:
# 데이터 검증 3
# 실제 Membership 변화가 감지된 월 수 확인

changed_months = monthly_changes[
    monthly_changes["changed"]
].copy()

print(
    "Changed periods:",
    len(changed_months)
)

changed_months

In [ ]:
#05a-8. 월별 탐색 결과 저장
# 데이터 검증 결과 저장
# 월별 KRX300 Membership 변경 구간 저장
# 목적:월말 Snapshot 기준으로 탐지된 변경 구간 보존.이후 Daily Point-in-Time Universe와 교차 검증

monthly_changes.to_csv(
    KRX_UNIVERSE_DIR / "krx300_monthly_membership_changes.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    KRX_UNIVERSE_DIR / "krx300_monthly_membership_changes.csv"
)

In [ ]:
#05a-9. daily point-in-time universe 수집
# Point-in-Time Universe 구축 1
# 모든 실제 거래일의 KRX300 구성종목 수집

# 목적:월말 Snapshot으로 놓칠 수 있는 월중 Membership 변화 방지, 각 거래일 당시 실제 KRX300 구성종목 보존
# 저장 방식:일정 간격으로 중간 저장, 이미 수집된 날짜는 재실행 시 건너뜀

DAILY_MEMBERSHIP_PATH = (
    KRX_UNIVERSE_DIR
    / "krx300_daily_membership_raw.csv"
)

In [ ]:
# Point-in-Time Universe 구축 2
# 기존 수집 데이터가 있으면 불러와 이어서 실행

if DAILY_MEMBERSHIP_PATH.exists():

    existing_daily_membership = pd.read_csv(
        DAILY_MEMBERSHIP_PATH,
        dtype={
            "ticker": str,
            "index_code": str
        },
        parse_dates=["date"]
    )

    collected_dates = set(
        existing_daily_membership["date"]
        .dt
        .normalize()
    )

    daily_membership_records = (
        existing_daily_membership
        .to_dict("records")
    )

    print(
        "Existing rows:",
        len(existing_daily_membership)
    )

    print(
        "Collected dates:",
        len(collected_dates)
    )

else:

    collected_dates = set()
    daily_membership_records = []

    print(
        "No existing daily membership file."
    )

In [ ]:
# 연결 상태 점검
# KRX 구성종목 API가 다시 정상 응답하는지 단일 날짜로 확인

test_members = get_krx300_members(
    "20260915"
)

print(
    "Count:",
    len(test_members)
)

print(
    test_members[:10]
)

In [ ]:
#05a-10. Point-in-Time Universe 정제 1
#Membership 변화가 확인된 월 구간 안에서 주간 검사 날짜 생성
#목적:2,112거래일 전체를 조회하지 않고 API 호출량 축소, 월별 변화 구간을 주 단위로 좁힘
#현재 상태:전체 104개월 중 Membership 변화 구간 48개 확인

weekly_refinement_records = []

for row in changed_months.itertuples():

    previous_date = pd.Timestamp(
        row.previous_date
    )

    current_date = pd.Timestamp(
        row.current_date
    )

    # 해당 월 구간의 실제 거래일만 추출
    interval_dates = trading_calendar[
        (trading_calendar > previous_date)
        & (trading_calendar <= current_date)
    ]

    interval_df = pd.DataFrame({
        "date": interval_dates
    })

    # 각 주의 마지막 실제 거래일 선택
    interval_df["week"] = (
        interval_df["date"]
        .dt
        .to_period("W-FRI")
    )

    weekly_dates = (
        interval_df
        .groupby("week")["date"]
        .max()
        .tolist()
    )

    # 구간 시작점도 포함
    candidate_dates = (
        [previous_date]
        + weekly_dates
        + [current_date]
    )

    candidate_dates = sorted(
        set(candidate_dates)
    )

    for date in candidate_dates:

        weekly_refinement_records.append({
            "monthly_previous_date": previous_date,
            "monthly_current_date": current_date,
            "check_date": date
        })


weekly_refinement_schedule = (
    pd.DataFrame(
        weekly_refinement_records
    )
    .drop_duplicates()
    .sort_values(
        [
            "monthly_previous_date",
            "check_date"
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Weekly check rows:",
    len(weekly_refinement_schedule)
)

print(
    "Unique check dates:",
    weekly_refinement_schedule[
        "check_date"
    ].nunique()
)

weekly_refinement_schedule.head(20)

In [ ]:
#05a-11. Point-in-Time Universe 정제 2
#이미 수집한 월말 Membership을 Cache로 등록
# 목적:이미 알고 있는 날짜를 KRX에 다시 요청하지 않음, API 호출 최소화

membership_cache = {
    pd.Timestamp(date): set(
        group["ticker"]
    )
    for date, group
    in monthly_membership.groupby("date")
}

print(
    "Cached monthly snapshots:",
    len(membership_cache)
)

In [ ]:
#05a-12. Point-in-Time Universe 정제 3
#주간/일간 추가 조회 결과를 저장할 Cache 파일 준비
# 목적:실행 중단 후 다시 시작 가능, 이미 성공한 API 요청 재호출 방지

REFINEMENT_CACHE_PATH = (
    KRX_UNIVERSE_DIR
    / "krx300_refinement_membership_cache.csv"
)

if REFINEMENT_CACHE_PATH.exists():

    refinement_cache_df = pd.read_csv(
        REFINEMENT_CACHE_PATH,
        dtype={
            "ticker": str
        },
        parse_dates=[
            "date"
        ]
    )

    for date, group in refinement_cache_df.groupby(
        "date"
    ):

        membership_cache[
            pd.Timestamp(date)
        ] = set(
            group["ticker"]
        )

    print(
        "Existing refinement cache loaded:",
        refinement_cache_df[
            "date"
        ].nunique(),
        "dates"
    )

else:

    refinement_cache_df = pd.DataFrame(
        columns=[
            "date",
            "ticker",
            "member_count"
        ]
    )

    print(
        "No existing refinement cache."
    )

In [ ]:
#05a-13. API 안전장치
# KRX300 Membership을 cache 우선으로 조회
# 원칙:
# 1. cache에 있으면 API 호출하지 않음
# 2. 새로운 날짜만 KRX 요청
# 3. 빈 결과가 오면 즉시 오류 발생
# 4. 요청 사이 최소 1초 대기

def get_membership_safe(date):

    date = pd.Timestamp(
        date
    ).normalize()

    # 이미 Cache에 있으면 바로 반환
    if date in membership_cache:

        return membership_cache[
            date
        ]

    date_str = date.strftime(
        "%Y%m%d"
    )

    members = get_krx300_members(
        date_str
    )

    members = set(
        members
    )

    # 정상적인 KRX300 결과가 아니면 중단
    if len(members) == 0:

        raise RuntimeError(
            f"KRX membership request failed: {date_str}"
        )

    # cache에 저장
    membership_cache[
        date
    ] = members

    # 파일에도 저장
    new_rows = pd.DataFrame({
        "date": [
            date
        ] * len(members),

        "ticker": sorted(
            members
        ),

        "member_count": [
            len(members)
        ] * len(members)
    })

    global refinement_cache_df

    refinement_cache_df = pd.concat(
        [
            refinement_cache_df,
            new_rows
        ],
        ignore_index=True
    )

    refinement_cache_df = (
        refinement_cache_df
        .drop_duplicates(
            subset=[
                "date",
                "ticker"
            ]
        )
    )

    refinement_cache_df.to_csv(
        REFINEMENT_CACHE_PATH,
        index=False,
        encoding="utf-8-sig"
    )

    # 서버 요청 간격 확보
    time.sleep(
        1.2
    )

    return members

In [ ]:
#05a-14. 주간 membership 탐색
# Point-in-Time Universe 정제 4
# 변화 월 구간을 주 단위로 재조회
# 목적: Membership 변화가 실제 어느 주에 발생했는지 탐색
# 주의: KRX 로그인 정상 확인 후 실행

weekly_check_dates = sorted(
    weekly_refinement_schedule[
        "check_date"
    ].unique()
)

print(
    "Weekly dates to check:",
    len(weekly_check_dates)
)


for i, date in enumerate(
    weekly_check_dates,
    start=1
):

    date = pd.Timestamp(
        date
    )

    members = get_membership_safe(
        date
    )

    print(
        f"[{i}/{len(weekly_check_dates)}]",
        date.strftime("%Y-%m-%d"),
        "members:",
        len(members)
    )


In [ ]:
#05a-15. 데이터 검증 및 정제
#주간 Snapshot 사이 Membership 변화 탐지
# 목적:일별 조회가 필요한 구간만 최종 선정

weekly_change_records = []


for (
    monthly_previous_date,
    group
) in weekly_refinement_schedule.groupby(
    "monthly_previous_date"
):

    check_dates = sorted(
        group[
            "check_date"
        ].unique()
    )

    for previous_date, current_date in zip(
        check_dates[:-1],
        check_dates[1:]
    ):

        previous_date = pd.Timestamp(
            previous_date
        )

        current_date = pd.Timestamp(
            current_date
        )

        previous_members = membership_cache[
            previous_date
        ]

        current_members = membership_cache[
            current_date
        ]

        added = (
            current_members
            - previous_members
        )

        removed = (
            previous_members
            - current_members
        )

        if added or removed:

            weekly_change_records.append({
                "previous_date": previous_date,
                "current_date": current_date,

                "added_count": len(
                    added
                ),

                "removed_count": len(
                    removed
                )
            })


weekly_change_intervals = pd.DataFrame(
    weekly_change_records
)

print(
    "Weekly change intervals:",
    len(weekly_change_intervals)
)

weekly_change_intervals.head(20)

In [ ]:
#05a-16. 변경된 주만 일별 검사 날짜 생성
# Point-in-Time Universe 정제 5
# Membership 변화가 확인된 주 구간만 일별 날짜 생성
# 목적:정확한 구성종목 변경 Effective Date 탐색

daily_refinement_dates = set()


for row in weekly_change_intervals.itertuples():

    previous_date = pd.Timestamp(
        row.previous_date
    )

    current_date = pd.Timestamp(
        row.current_date
    )

    interval_dates = trading_calendar[
        (trading_calendar > previous_date)
        & (trading_calendar <= current_date)
    ]

    for date in interval_dates:

        daily_refinement_dates.add(
            pd.Timestamp(date)
        )


daily_refinement_dates = sorted(
    daily_refinement_dates
)

print(
    "Daily dates to check:",
    len(daily_refinement_dates)
)

daily_refinement_dates[:20]

In [ ]:
#05a-17. 변경 주의 일별 membership 조회
# Point-in-Time Universe 정제 6
# 변경이 확인된 주의 거래일별 Membership 조회
# 목적: 실제 KRX300 구성 변경일 정확히 탐색

for i, date in enumerate(
    daily_refinement_dates,
    start=1
):

    members = get_membership_safe(
        date
    )

    print(
        f"[{i}/{len(daily_refinement_dates)}]",
        date.strftime("%Y-%m-%d"),
        "members:",
        len(members)
    )

In [ ]:
#05a-18. Point-in-Time Universe 정제 7
# 연속 거래일 Membership 비교 후 정확한 변경일 추출
# effective Date: 전 거래일과 구성종목 목록이 처음 달라진 날짜

membership_change_events = []

for row in weekly_change_intervals.itertuples():

    previous_date = pd.Timestamp(
        row.previous_date
    )

    current_date = pd.Timestamp(
        row.current_date
    )

    interval_dates = trading_calendar[
        (trading_calendar >= previous_date)
        & (trading_calendar <= current_date)
    ]

    interval_dates = sorted(
        interval_dates
    )

    for prev_date, curr_date in zip(
        interval_dates[:-1],
        interval_dates[1:]
    ):

        prev_date = pd.Timestamp(
            prev_date
        )

        curr_date = pd.Timestamp(
            curr_date
        )

        # 필요한 날짜만 비교
        if (
            prev_date not in membership_cache
            or curr_date not in membership_cache
        ):
            continue

        previous_members = membership_cache[
            prev_date
        ]

        current_members = membership_cache[
            curr_date
        ]

        added = (
            current_members
            - previous_members
        )

        removed = (
            previous_members
            - current_members
        )

        if added or removed:

            membership_change_events.append({
                "previous_date": prev_date,
                "effective_date": curr_date,

                "added_count": len(
                    added
                ),

                "removed_count": len(
                    removed
                ),

                "added_tickers": ",".join(
                    sorted(added)
                ),

                "removed_tickers": ",".join(
                    sorted(removed)
                )
            })


membership_change_events = (
    pd.DataFrame(
        membership_change_events
    )
    .drop_duplicates(
        subset=[
            "effective_date"
        ]
    )
    .sort_values(
        "effective_date"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Detected change events:",
    len(membership_change_events)
)

membership_change_events.head(20)

In [ ]:
#05a-19. krx 300 membership 변경 event 저장

MEMBERSHIP_EVENTS_PATH = (
    CLEAN_DIR
    / "krx300_membership_change_events.csv"
)

membership_change_events.to_csv(
    MEMBERSHIP_EVENTS_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    MEMBERSHIP_EVENTS_PATH
)

In [ ]:
# 보류
# KRX IP 이용 제한으로 인해 KRX 관련 API 호출 중단
# - pykrx 사용 중단
# - KRX Open API 사용 중단
# - 제한 해제 후 공식 경로만 재검토

지금한거:
KRX300 index
→ Trading Calendar
→ 월별 Membership
→ Membership 변화 탐지
→ Point-in-Time Universe 구축 시도
→ KRX 제한으로 이후 보류

05C-20~27
공식 API 연결
↓
하루치 Raw 수집
↓
KOSPI + KOSDAQ 결합
↓
컬럼 / 코드 / dtype 검증
────────────────────
여기까지 지금 실행

다음

05C-28
컬럼명 표준화

05C-29
숫자형 변환

05C-30
결측 / 중복 검사

05C-31
가격 이상값 검증

05C-32
KRX300 Membership과 ticker 연결 테스트

05C-33
기간 수집 전략 확정

05C-34
2018~2026 Raw Market Data 수집

In [ ]:
# KRX Universe Pipeline - 현재 보류
# 진행 완료:
# - KRX300 Trading Calendar 생성
# - 월별 Membership Snapshot 수집
# - Membership 변경 구간 탐지

# 보류 사유:KRX Data Marketplace 자동화 대량 조회로 IP 일시 제한

# 향후:
# - 공식 경로를 이용해 Point-in-Time Membership 보완
# - 최종 Universe 검증 후 05D Common Dataset에서 결합